# Tableau

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/tableau.ipynb)

Tableau is a visual analytics platform: you model a data source, build worksheets by dragging fields onto shelves, and publish governed dashboards with extracts, schedules and permissions. This notebook covers what a data engineer needs: grain, extracts versus live connections, how Tableau generates SQL, LOD expressions, filters, row-level security and performance. Tableau itself is not scriptable here, so the examples reproduce its query semantics in DuckDB.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.0", "pandas": "pandas>=2.2"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## Architecture and products

| Component | Role |
|---|---|
| Tableau Desktop | Authoring. Workbooks (`.twb`) and data sources (`.tds`) are XML; packaged versions (`.twbx`, `.tdsx`) bundle extracts and files. |
| Tableau Prep | Visual data preparation flows (`.tfl`). Useful for analysts; data engineers usually prefer dbt or SQL in the warehouse. |
| Tableau Server / Cloud | Hosting: VizQL renders views, Backgrounder runs extract refreshes and subscriptions, the Repository (PostgreSQL) stores content and permissions, the Gateway routes requests. Cloud is the SaaS version. |
| Tableau Bridge | Keeps on-premises sources reachable from Tableau Cloud. |
| REST API, Metadata API, Hyper API, `tabcmd` | Automation: publish, refresh, permissions, lineage, building `.hyper` files in a pipeline. |
| Tableau Public / Reader | Free viewing; not for governed company data. |

Licensing is per role: **Creator** (author and publish), **Explorer** (edit on published sources), **Viewer** (consume). The license mix is usually the largest cost line. When sizing a Server deployment, user count alone is not enough: capacity depends on concurrent views, extract count and size, and the refresh window that Backgrounder must fit into.

The durable difference from other BI tools is *where the semantic layer lives*. Looker and dbt keep metric definitions in versioned code; Power BI keeps them in a tabular model; Tableau keeps most of them in workbooks and published data sources, which are fast to author but hard to diff. That is the argument for letting a **certified published data source**, not individual workbooks, own the definition of revenue or margin.

### Where you'll use it

- **Executive dashboards:** a sales leader explores pipeline by region on a published Tableau dashboard, with filters and drill-down.

## Modeling data for Tableau: grain first

Tableau aggregates whatever rows survive the joins; it cannot know that a sum is now double counting. Treat every published data source as a contract about grain, written next to it: "one row per order line per day".

Rules that survive an audit:

+ Publish one certified data source per domain instead of dozens of workbook-level connections.
+ Keep joins in the warehouse; connect Tableau to marts already conformed at a known grain.
+ Expose conformed dimensions (`dim_date`, `dim_customer`) so date and customer logic is identical everywhere.
+ Keep a line-level `Detail` source for drill-through instead of hiding detail behind a slow live join.

The classic bug is **fan-out**: joining a header table (orders) to a child table (items) repeats each order once per item, so `SUM(order_amount)` inflates. The next cell reproduces it and shows the two standard fixes.

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("""
CREATE TABLE orders AS SELECT * FROM (VALUES
    (1, 'EMEA', 400.0), (2, 'EMEA', 600.0), (3, 'US', 500.0)
) t(order_id, region, order_amount);
CREATE TABLE order_items AS SELECT * FROM (VALUES
    (1, 'A'), (1, 'B'), (1, 'C'), (2, 'A'), (2, 'B'), (2, 'C'), (3, 'A')
) t(order_id, sku);
""")

true_total = con.sql("SELECT SUM(order_amount) FROM orders").fetchall()[0][0]
fanned_out = con.sql("""
    SELECT SUM(o.order_amount) FROM orders o JOIN order_items i USING (order_id)
""").fetchall()[0][0]

# Fix 1: aggregate the many side to the header grain before joining.
pre_aggregated = con.sql("""
    WITH items AS (SELECT order_id, COUNT(*) AS item_count FROM order_items GROUP BY order_id)
    SELECT SUM(o.order_amount) FROM orders o JOIN items USING (order_id)
""").fetchall()[0][0]

# Fix 2: the SQL equivalent of {FIXED [order_id] : MIN([order_amount])}, then SUM.
fixed_lod = con.sql("""
    WITH joined AS (SELECT o.order_id, o.order_amount FROM orders o JOIN order_items USING (order_id)),
         per_order AS (SELECT order_id, MIN(order_amount) AS amount FROM joined GROUP BY order_id)
    SELECT SUM(amount) FROM per_order
""").fetchall()[0][0]

print(f"true={true_total} fanned_out={fanned_out} pre_aggregated={pre_aggregated} fixed_lod={fixed_lod}")
assert fanned_out == 3500.0 and true_total == pre_aggregated == fixed_lod == 1500.0

A small sample often hides fan-out because it happens not to contain multi-item orders. Test totals against the source of truth (`SUM` on the header table) as part of the data source certification.

## Joins, relationships and blending

| Mechanism | Where it runs | Grain behaviour | Use when |
|---|---|---|---|
| **Physical join** (Data Source canvas) | One SQL query at the source | Fixed for every view; can fan out | Tables share a grain or the join is one-to-one/many-to-one |
| **Relationship** (logical layer, 2020.2+) | Tableau decides per view which tables to join | Joins only the tables the view uses, aggregates each at its native grain; avoids fan-out | Default for multi-fact or mixed-grain models |
| **Blend** | Two queries; secondary aggregated to the link and left-joined on the client | Secondary measures exist only at the link grain | Sources cannot be joined in the warehouse (different systems) |

Relationship tips: set cardinality and referential integrity explicitly, and enforce uniqueness and foreign keys in the warehouse so the assumption is testable. **Assume Referential Integrity** lets Tableau drop a join when no field from the other table is used (join culling); if the integrity does not actually hold, totals change depending on which fields a view uses.

A blend repeats the secondary value for each primary mark that shares the link value, so re-aggregating it across marks double counts:

In [ ]:
con.execute("""
CREATE TABLE opportunities AS SELECT * FROM (VALUES
    ('opp1', 'acct1', 100.0), ('opp2', 'acct1', 300.0), ('opp3', 'acct2', 200.0)
) t(opportunity_id, account_id, pipeline);
CREATE TABLE marketing_spend AS SELECT * FROM (VALUES
    ('acct1', 'search', 50.0), ('acct1', 'social', 30.0), ('acct2', 'search', 40.0)
) t(account_id, campaign, spend);
""")

# A blend: aggregate the secondary to the link field, then left join on the client.
blend = con.sql("""
    WITH secondary AS (SELECT account_id, SUM(spend) AS spend FROM marketing_spend GROUP BY account_id)
    SELECT p.opportunity_id, p.account_id, p.pipeline, s.spend
    FROM opportunities p LEFT JOIN secondary s USING (account_id)
    ORDER BY p.opportunity_id
""").df()
print(blend)

true_spend = con.sql("SELECT SUM(spend) FROM marketing_spend").fetchall()[0][0]
summed_across_marks = blend["spend"].sum()
print(f"true spend={true_spend}, spend summed across opportunity marks={summed_across_marks}")
assert true_spend == 120.0 and summed_across_marks == 200.0

## Live connections versus extracts

**Live**: every view change sends a query to the source. Data is fresh; source latency, concurrency limits and warehouse permissions all apply. **Extract**: Tableau runs the source query once and stores the result as a `.hyper` file (columnar, dictionary- and run-length-encoded); later views are answered by the Hyper engine. An extract stores rows after extract filters, the schema, and field metadata. It does **not** store indexes, views, or warehouse permissions.

```text
Decision rule
  freshness need shorter than the refresh interval?   -> live (or shorten the schedule)
  source cannot tolerate dashboard concurrency?        -> extract
  dashboard scans billions of rows?                    -> extract or pre-aggregate, never raw live
  row-level security enforced in the warehouse?        -> live, so the policies apply
```

A common compromise: an hourly "hot" extract of executive aggregates plus live access for drill-through.

Refresh types:

+ **Full**: rebuild the snapshot. Predictable, expensive, picks up updates and deletes.
+ **Incremental**: append rows where the increment key (a date or ID) is greater than the last stored value. Cheap, but it never sees updated or deleted rows, so schedule a periodic full refresh (for example weekly) to correct drift.

The next cell simulates both behaviours: a late update to an old row is missed by incremental refresh until the full rebuild.

### Where you'll use it

- **Choosing a connection:** an extract gives fast dashboards on a nightly snapshot; a live connection is used when the data must be current to the minute.

In [ ]:
con.execute("""
CREATE TABLE sales_source AS SELECT * FROM (VALUES
    (DATE '2026-10-01', 'EMEA', 100.0), (DATE '2026-10-02', 'US', 200.0)
) t(order_date, region, revenue);
CREATE TABLE sales_extract AS SELECT * FROM sales_source;
""")

# New rows arrive, and an old row is corrected in the source.
con.execute("INSERT INTO sales_source VALUES (DATE '2026-10-03', 'EMEA', 50.0)")
con.execute("UPDATE sales_source SET revenue = 120.0 WHERE order_date = DATE '2026-10-01'")


def total(table: str) -> float:
    return con.sql(f"SELECT SUM(revenue) FROM {table}").fetchall()[0][0]


live_total = total("sales_source")
stale_total = total("sales_extract")

# Incremental refresh: append rows beyond the stored high-water mark only.
con.execute("""
INSERT INTO sales_extract
SELECT * FROM sales_source
WHERE order_date > (SELECT MAX(order_date) FROM sales_extract)
""")
incremental_total = total("sales_extract")

con.execute("CREATE OR REPLACE TABLE sales_extract AS SELECT * FROM sales_source")
full_total = total("sales_extract")

print(f"live={live_total} stale={stale_total} incremental={incremental_total} full={full_total}")
assert (live_total, stale_total, incremental_total, full_total) == (370.0, 300.0, 350.0, 370.0)

Refreshes are normally triggered by the orchestrator right after the warehouse load, not by a fixed clock schedule, so the extract never refreshes before the data lands:

```bash
tabcmd login -s https://tableau.example.com -t finance -u svc_tableau --password-file /secrets/tableau
tabcmd refreshextracts --project "Finance" --datasource "Certified Sales" --synchronous
tabcmd logout
```

The REST API (or the `tableauserverclient` Python library) does the same from Airflow: `POST /api/<version>/sites/<site>/datasources/<id>/refresh` returns a job ID to poll. Building extracts outside Tableau is done with the Hyper API:

```python
from tableauhyperapi import Connection, CreateMode, HyperProcess, TableDefinition, Telemetry

table = TableDefinition("Extract", [
    TableDefinition.Column("order_id", "text"),
    TableDefinition.Column("order_date", "date"),
    TableDefinition.Column("revenue", "double precision"),
])
with HyperProcess(telemetry=Telemetry.DO_NOT_SEND_USAGE_DATA_TO_TABLEAU) as hyper:
    with Connection(hyper.endpoint, "orders.hyper", CreateMode.CREATE_AND_REPLACE) as conn:
        conn.catalog.create_schema("Extract")
        conn.catalog.create_table(table)
        conn.execute_command("COPY \"Extract\".\"Extract\" FROM 'orders.csv' WITH (format csv, header true)")
```

### Designing a small, fast extract

+ Hide unused fields before extracting ("Hide All Unused Fields"); each column costs storage and refresh time.
+ Use extract filters to drop history nobody queries.
+ Aggregate to visible dimensions ("Aggregate data for visible dimensions") when detail is not needed.
+ Prefer integer or date keys over long strings; materialize expensive calculations in the warehouse.

A rough size estimate: numbers and dates cost about 8 bytes per row, dictionary-encoded strings a small index per row plus a dictionary of distinct values.

In [ ]:
ROWS = 10_000_000
AVG_STRING_BYTES = 12
FIXED_WIDTH_BYTES = 8
DICTIONARY_INDEX_BYTES = 4
COLUMNS = [
    ("order_id", "string", 10_000_000),
    ("order_date", "date", 2_920),
    ("customer_id", "string", 1_200_000),
    ("product_id", "string", 90_000),
    ("region", "string", 12),
    ("unit_price", "numeric", 0),
    ("quantity", "numeric", 0),
    ("net_revenue", "numeric", 0),
    ("cost", "numeric", 0),
    ("margin", "numeric", 0),
]


def extract_bytes(rows: int, columns: list[tuple[str, str, int]]) -> int:
    per_row = dictionary = 0
    for _, kind, distinct in columns:
        if kind == "string":
            per_row += DICTIONARY_INDEX_BYTES
            dictionary += distinct * AVG_STRING_BYTES
        else:
            per_row += FIXED_WIDTH_BYTES
    return rows * per_row + dictionary


with_order_id = extract_bytes(ROWS, COLUMNS)
without_order_id = extract_bytes(ROWS, [c for c in COLUMNS if c[0] != "order_id"])
print(f"all columns:      {with_order_id / 1024**3:.2f} GiB")
print(f"order_id hidden:  {without_order_id / 1024**3:.2f} GiB")
assert with_order_id == 775_480_144
assert without_order_id < with_order_id * 0.8

## How a worksheet becomes SQL

+ **Dimensions** (blue headers when discrete) become the `GROUP BY`; **measures** are aggregated inside it with their default aggregation (`SUM`, `AVG`, `COUNTD`, `MIN`/`MAX`, or none for row level).
+ **Discrete** fields create headers; **continuous** (green) fields create axes. `MONTH(date)` as discrete gives 12 headers (date part); as continuous it gives a timeline (date value). Many "my line chart looks wrong" questions are this toggle.
+ **Shelves**: every field on Rows, Columns, Color, Size or Detail joins the `GROUP BY`. Dropping `Order ID` on Detail just for a tooltip turns a 96-row aggregate into a row-level scan.
+ **Aggregate Measures** off (Analysis menu) issues a row-level query: one mark per record.
+ Numeric IDs should be dimensions (summing `Customer ID` is meaningless); count them with `COUNTD`.
+ `ATTR(x)` returns the value if it is unique in the partition and `*` otherwise; it reveals fan-out, it does not fix it.

The query for `SUM(Net Revenue)` by Category and Month looks like this, and the row count depends on the dimensions, not on the size of the fact table:

In [ ]:
import pandas as pd

con.execute("""
CREATE TABLE fact_sales AS
SELECT
    DATE '2026-01-01' + (i % 365) * INTERVAL 1 DAY AS order_date,
    ['Bikes', 'Clothing', 'Accessories'][1 + i % 3] AS category,
    ['EMEA', 'US', 'APAC', 'LATAM'][1 + (i // 3) % 4] AS region,
    'C' || (i % 500) AS customer_id,
    ROUND(10 + (i * 37) % 90, 2) AS net_revenue
FROM range(100000) r(i)
""")

view = con.sql("""
    SELECT category, DATE_TRUNC('month', order_date) AS month, SUM(net_revenue) AS net_revenue
    FROM fact_sales
    GROUP BY 1, 2
""").df()
print(f"source rows=100,000 -> marks={len(view)}")
assert len(view) == 3 * 12

### Ratio of sums, not average of ratios

A margin or revenue-per-order KPI must be `SUM(numerator) / SUM(denominator)` so it re-aggregates correctly at every level of the view. Averaging a row-level ratio weights small groups the same as large ones.

In [ ]:
kpi = pd.DataFrame({"region": ["East", "West"], "orders": [10, 90], "revenue": [1_000, 180_000]})

ratio_of_sums = kpi["revenue"].sum() / kpi["orders"].sum()
average_of_ratios = (kpi["revenue"] / kpi["orders"]).mean()
print(f"SUM(revenue)/SUM(orders) = {ratio_of_sums:,.1f}")
print(f"AVG(revenue/orders)      = {average_of_ratios:,.1f}")
assert round(ratio_of_sums, 1) == 1810.0 and round(average_of_ratios, 1) == 1050.0

## Calculations: row-level, aggregate, LOD and table calculations

| Type | Example | Where it runs |
|---|---|---|
| Row-level | `[Revenue] - [Cost]` | In the source query, per row |
| Aggregate | `SUM([Revenue] - [Cost]) / SUM([Revenue])` | In the source `GROUP BY` |
| LOD expression | `{FIXED [Customer] : MIN([Order Date])}` | Subquery joined back at a declared grain |
| Table calculation | `RANK(SUM([Revenue]))`, `RUNNING_SUM(...)` | In Tableau, over the result set already returned |

LOD expressions decouple a calculation's grain from the view's grain:

+ `FIXED` computes at exactly the listed dimensions, ignoring view dimensions. It is evaluated **before** dimension filters (only context, data source and extract filters apply).
+ `INCLUDE` adds dimensions to the view grain (for example, average of per-customer totals by region).
+ `EXCLUDE` removes dimensions from the view grain (for example, region share of the total).

In SQL these are a grouped subquery joined back, or a window function:

In [ ]:
lod = con.sql("""
    WITH customer_totals AS (            -- {INCLUDE [customer_id] : SUM([net_revenue])}
        SELECT region, customer_id, SUM(net_revenue) AS customer_revenue
        FROM fact_sales GROUP BY 1, 2
    ),
    first_order AS (                     -- {FIXED [customer_id] : MIN([order_date])}
        SELECT customer_id, MIN(order_date) AS first_order_date FROM fact_sales GROUP BY 1
    )
    SELECT
        region,
        SUM(customer_revenue) AS revenue,
        AVG(customer_revenue) AS avg_revenue_per_customer,
        SUM(customer_revenue) / SUM(SUM(customer_revenue)) OVER () AS share_of_total,  -- EXCLUDE [region]
        COUNT(DISTINCT CASE WHEN fo.first_order_date >= DATE '2026-07-01' THEN customer_id END) AS new_customers_h2
    FROM customer_totals
    JOIN first_order fo USING (customer_id)
    GROUP BY region
    ORDER BY region
""").df()
print(lod)
assert abs(lod["share_of_total"].sum() - 1.0) < 1e-9

### Table calculations: rank, running total, top and bottom N

Table calculations run on the aggregated result Tableau received, scoped by "Compute Using" (table across, pane down, specific dimensions). Because they run after the query, a filter on a table calculation hides marks but does not reduce what the source returned. Showing the top five and bottom five in one view is a rank-based calculated field:

```text
// Calculated field [Top or Bottom 5], placed on Filters = True
RANK(SUM([Net Revenue])) <= 5 OR RANK(SUM([Net Revenue]), 'asc') <= 5
```

`RANK` gives ties the same rank and skips (1, 2, 2, 4); `RANK_DENSE` does not skip; `RANK_UNIQUE` breaks ties by order; `INDEX()` is the row position. The SQL equivalent:

In [ ]:
ranked = con.sql("""
    WITH by_customer AS (
        SELECT customer_id, SUM(net_revenue) AS revenue FROM fact_sales GROUP BY 1
    )
    SELECT customer_id, revenue,
           RANK() OVER (ORDER BY revenue DESC) AS rank_desc,
           RANK() OVER (ORDER BY revenue ASC) AS rank_asc,
           SUM(revenue) OVER (ORDER BY revenue DESC, customer_id) AS running_revenue
    FROM by_customer
    QUALIFY rank_desc <= 5 OR rank_asc <= 5
    ORDER BY revenue DESC
""").df()
print(ranked)
assert len(ranked) >= 10

## Parameters, sets, groups and bins

+ **Parameter**: a single typed workspace value (number, string, date, boolean) with an "all", list or range domain. It is substituted into calculations, reference lines, bins and top-N logic, and can be set by a parameter action. It is a scalar, so it cannot vary per row. Use a real filter when a filter works; a parameter-driven calculated filter usually cannot be pushed down as efficiently.
+ **Set**: marks members of a dimension IN or OUT. Constant sets are a fixed list; computed sets (condition or top-N) update with the data. Sets can be filters, can be compared in calculations, and power set actions ("highlight selected customers").
+ **Group**: relabels members of one dimension into buckets (60 countries into 6 regions). Static metadata; anything that changes (top-10 customers) belongs in a computed set or in the warehouse.
+ **Bin**: splits a continuous measure into equal-width buckets for histograms.

```text
// Parameter [Top N] (integer, default 10) drives a dynamic top-N filter
RANK(SUM([Net Revenue])) <= [Top N]
```

## Filters and the order of operations

Tableau evaluates filters in a fixed order, and the order decides what each calculation sees:

1. Extract filters (applied when the extract is built)
2. Data source filters (applied to every query on the source; also how you restrict a published source)
3. Context filters (create a temporary result that later filters and top-N sets work within)
4. Sets, conditional and top-N filters, and `FIXED` LOD expressions
5. Dimension filters
6. `INCLUDE`/`EXCLUDE` LOD expressions, then measure filters
7. Totals, forecasts, then table calculation filters (which only hide marks)

The practical consequences: a `FIXED` LOD ignores a normal dimension filter unless that filter is added to context; a "Top 10 customers" set computed before a region filter returns the global top ten, not the top ten in the region, unless the region filter is a context filter. For cost, push restrictions as early as possible: extract and data source filters reduce what every query scans, while table calculation filters reduce nothing at the source.

In [ ]:
FILTERS = [
    ("measure filter: SUM(margin) > 1000", "measure"),
    ("context filter: region IN (EMEA, US)", "context"),
    ("extract filter: order_date >= 2018-01-01", "extract"),
    ("table calculation filter: RANK(SUM(revenue)) <= 20", "table_calculation"),
    ("dimension filter: category = 'Bikes'", "dimension"),
    ("data source filter: tenant = 'acme'", "data_source"),
]
EVALUATION_ORDER = ["extract", "data_source", "context", "dimension", "measure", "table_calculation"]

ordered = sorted(FILTERS, key=lambda item: EVALUATION_ORDER.index(item[1]))
for position, (name, _) in enumerate(ordered, 1):
    print(f"{position}. {name}")
assert [kind for _, kind in ordered] == EVALUATION_ORDER

The top-N interaction, reproduced: the global top three customers are not the top three in EMEA unless region is applied first (as context).

In [ ]:
global_top = con.sql("""
    SELECT customer_id FROM fact_sales GROUP BY 1 ORDER BY SUM(net_revenue) DESC, 1 LIMIT 3
""").df()["customer_id"].tolist()

# Region as a normal dimension filter: the top-N set was computed on all regions first.
set_then_filter = con.sql(f"""
    SELECT DISTINCT customer_id FROM fact_sales
    WHERE region = 'EMEA' AND customer_id IN {tuple(global_top)}
""").df()["customer_id"].tolist()

# Region as a context filter: the top-N is computed inside EMEA.
context_then_set = con.sql("""
    SELECT customer_id FROM fact_sales WHERE region = 'EMEA'
    GROUP BY 1 ORDER BY SUM(net_revenue) DESC, 1 LIMIT 3
""").df()["customer_id"].tolist()

print("global top 3:", global_top)
print("normal filter keeps:", sorted(set_then_filter))
print("context filter gives:", context_then_set)
assert len(context_then_set) == 3 and len(set_then_filter) < 3

## Row-level security

Options, from most to least governable:

1. **Warehouse RLS with a live connection** (Snowflake row access policies, BigQuery row-level security), passing the viewer's identity through (OAuth, impersonation or initial SQL). One rule for every tool.
2. **Entitlement table in the data source**: join the fact to `user_entitlements(username, region)` and add a data source filter `[username] = USERNAME()`. Works for extracts and scales to many users; keep the entitlement table small and the join on an integer key.
3. **User filter / `ISMEMBEROF('group')` calculations** in a published data source. Fine for a handful of groups; hard to audit at scale.
4. **Virtual connections with data policies** (Data Management add-on) centralize the policy for several sources.

An extract does not inherit warehouse RLS: if security lives in the warehouse and you switch to an extract, every viewer sees every row unless you recreate the rule in Tableau. Apply the filter as a **data source filter on the published source** so workbook authors cannot remove it. The entitlement pattern in SQL, with `USERNAME()` as a parameter:

### Where you'll use it

- **Partner portals:** user filters limit each external partner to their own accounts.

In [ ]:
con.execute("""
CREATE TABLE user_entitlements AS SELECT * FROM (VALUES
    ('alice@acme.com', 'EMEA'), ('bob@acme.com', 'US'),
    ('auditor@acme.com', 'EMEA'), ('auditor@acme.com', 'US'),
    ('auditor@acme.com', 'APAC'), ('auditor@acme.com', 'LATAM')
) t(username, region)
""")


def revenue_visible_to(username: str) -> pd.DataFrame:
    return con.execute("""
        SELECT f.region, SUM(f.net_revenue) AS revenue
        FROM fact_sales f
        JOIN user_entitlements e ON e.region = f.region AND e.username = ?
        GROUP BY 1 ORDER BY 1
    """, [username]).df()


for user in ["alice@acme.com", "bob@acme.com", "auditor@acme.com", "mallory@acme.com"]:
    visible = revenue_visible_to(user)
    print(f"{user:18s} regions={visible['region'].tolist()}")

assert revenue_visible_to("alice@acme.com")["region"].tolist() == ["EMEA"]
assert revenue_visible_to("mallory@acme.com").empty  # no entitlement row: deny by default
assert revenue_visible_to("auditor@acme.com")["revenue"].sum() == con.sql(
    "SELECT SUM(net_revenue) FROM fact_sales").fetchall()[0][0]

## Performance

### Diagnosing a slow workbook

Start with a **Performance Recording** (Desktop: Help > Settings and Performance; Server: the View Performance admin view). Classify time by stage before changing anything:

| Dominant stage | Typical cause | Fix |
|---|---|---|
| Executing query | Heavy live query, custom SQL blocking pushdown, high-cardinality GROUP BY | Extract or pre-aggregated mart, data source filters, remove fields from Detail |
| Computing layout / rendering | Too many marks, many sheets on one dashboard | Fewer marks, fewer sheets, aggregate first |
| Compiling query | Very complex calculations, many LODs | Materialize calculations upstream |
| Connecting / blending | Slow connection, many blends | Join in the warehouse, publish one source |

In [ ]:
recording = [
    {"sheet": "Revenue", "stage": "query", "seconds": 4.2},
    {"sheet": "Revenue", "stage": "render", "seconds": 0.3},
    {"sheet": "KPI", "stage": "query", "seconds": 0.1},
    {"sheet": "KPI", "stage": "render", "seconds": 1.8},
]

by_sheet: dict[str, dict[str, float]] = {}
for event in recording:
    stages = by_sheet.setdefault(event["sheet"], {})
    stages[event["stage"]] = stages.get(event["stage"], 0.0) + event["seconds"]

bottlenecks = {sheet: max(stages, key=stages.get) for sheet, stages in by_sheet.items()}
print(bottlenecks)
assert bottlenecks == {"Revenue": "query", "KPI": "render"}

### Reducing query cost

+ **Pre-aggregate to the dashboard grain.** A 250M-row fact at 730 days x 40 categories x 12 regions needs only 350,400 rows.
+ **Avoid custom SQL** for anything a view can express: Tableau wraps custom SQL as a subquery and applies filters around it, which blocks many optimizations.
+ **Filters**: prefer data source and extract filters; use context filters to shrink a large set once for several dependent filters; avoid "Only Relevant Values" quick filters on huge dimensions (each one runs its own query); prefer action filters.
+ **Dashboards**: fewer sheets per dashboard, fixed size (enables cache reuse), hide unused fields, avoid long string calculations at row level.
+ **Caching**: Server caches query results per data source; identical queries from different viewers hit the cache unless user-specific filters (RLS) make each query unique. Extract refreshes invalidate the cache.

In [ ]:
FACT_ROWS = 250_000_000
grain_rows = 730 * 40 * 12
reduction = FACT_ROWS / grain_rows
print(f"aggregated rows = {grain_rows:,}; reduction factor = {reduction:,.0f}x")
assert grain_rows == 350_400 and reduction > 700

### Load testing

Every rendered view opens a VizQL session and runs queries, so load testing measures how many concurrent sessions the cluster sustains. **TabJolt** replays recorded interactions at a target concurrency; scripted REST API calls test exports and refreshes. Test after upgrades and before large rollouts, watch VizQL and Backgrounder CPU and memory, and keep extract refreshes out of peak viewing hours because Backgrounder competes for the same nodes.

## Publishing, embedding and dashboards

+ **Published versus embedded data sources**: a published source lives on Server once, with its own permissions, refresh schedule and certification, and many workbooks connect to it. An embedded source is private to one workbook and duplicates logic and refreshes. Publish anything shared.
+ **Dashboards** combine sheets, filters, actions (filter, highlight, URL, set, parameter) and layout containers. A **story** is a sequence of sheets or dashboards that walks through a narrative.
+ **Embedding**: Tableau's Embedding API v3 (`<tableau-viz>` web component) with Connected Apps (JWT signed by your app) is the current approach; older deployments use trusted tickets. With either, the user identity in the token drives `USERNAME()` and therefore row-level security.

```python
# Connected App: your backend signs a short-lived JWT; the browser passes it to <tableau-viz token="...">
import datetime, uuid, jwt  # PyJWT

token = jwt.encode(
    {
        "iss": CONNECTED_APP_CLIENT_ID,
        "exp": datetime.datetime.now(datetime.UTC) + datetime.timedelta(minutes=5),
        "jti": str(uuid.uuid4()),
        "aud": "tableau",
        "sub": "viewer@acme.com",
        "scp": ["tableau:views:embed"],
    },
    CONNECTED_APP_SECRET_VALUE,
    algorithm="HS256",
    headers={"kid": CONNECTED_APP_SECRET_ID, "iss": CONNECTED_APP_CLIENT_ID},
)
```

## When not to use Tableau

+ Metrics must be reused by applications and other tools: a code-based semantic layer (dbt Semantic Layer, Looker, Cube) fits better.
+ Heavy embedded, multi-tenant analytics at low cost per viewer.
+ Pixel-perfect operational reports and bursting (paginated reporting tools are better).
+ Teams that need workbooks under code review: `.twb` XML diffs are noisy.

## Practice scenarios

+ **Retail category management**: certified daily-grain extract, refreshed by Airflow after the warehouse load, with a line-level detail source for drill-through.
+ **Fintech regulatory reporting**: live connection to a warehouse with row access policies and OAuth pass-through, so access rules exist in one place.
+ **Marketing spend efficiency**: spend and conversions land in one warehouse model at the campaign-day grain instead of being blended.
+ **SaaS adoption for customer success**: entitlement-table RLS by account owner, embedded in the CRM with a Connected App.

Questions to check yourself:

1. Why can a `FIXED` LOD ignore your region filter, and how do you change that?
2. When does incremental extract refresh return wrong numbers?
3. Why does an extract weaken warehouse-enforced security?
4. What is the difference between a relationship and a physical join for a model with two fact tables?

### Where you'll use it

- **Self-service BI:** analysts publish governed workbooks to a shared site, and the data team owns the certified data source behind them.

### Where you'll use it

- **Operational or transactional reporting:** a workflow that needs writes back to the source belongs in an application, not a BI tool.